In [0]:
dbutils.widgets.text('catalog_name','')
dbutils.widgets.text('schema','')


In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema = dbutils.widgets.get('schema')

In [0]:

spark.sql(f"""create schema if not exists {catalog_name}.metadata;""")


In [0]:
spark.sql(f"""
create table if not exists {catalog_name}.metadata.tables (
table_id int,
table_name string,
source_system string, -- sqlserver / blob
source_schema string, -- dbo (null for blob)
source_table string,  -- table name (null for blob)
source_path string,
target_layer string, -- silver/gold
bronze_schema string,
silver_schema string,
gold_schema string,
active_flag string,
load_order int,
created_at timestamp
)
using delta""")

In [0]:
spark.sql(f"""
create table if not exists {catalog_name}.metadata.table_parameters(
table_id int,
parameter_name string,-- load_type / primary_key / watermark_column
parameter_value string,
created_at timestamp

)
using delta""")

In [0]:
spark.sql(f"""
create table if not exists {catalog_name}.metadata.table_watermarks(
table_id int,
last_watermark_value string,
last_updated_at timestamp,    
last_run_id bigint
)
using delta
PARTITIONED BY (table_id)""")

In [0]:
spark.sql(f"""
create table if not exists {catalog_name}.metadata.pipeline_runs(
run_id bigint, 
table_id int,
layer string,
start_time timestamp,
end_time timestamp,
status string,
number_of_records bigint,
error_message string
)
using delta
partitioned by (table_id)
""")

In [0]:
spark.sql(f"""
create schema if not exists {catalog_name}.source""")

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.source.volume""")

In [0]:
from pyspark.sql.functions import *
spark.sql(f"insert into {catalog_name}.metadata.tables value (1,'customer','sqlserver','banking','customers',NULL,'silver','bronze','silver',NULL,TRUE,1,current_timestamp())")

In [0]:
spark.sql(f"insert into {catalog_name}.metadata.table_parameters values (1,'load_type','MERGE',current_timestamp()),(1,'primary_key','customer_id',current_timestamp()),(1,'watermark_column','updated_at',current_timestamp())")

In [0]:
spark.sql(f"insert into {catalog_name}.metadata.table_watermarks values (1,'2022-01-01 00:00:00',current_timestamp(),NULL)")